# Customer Support Ticket Priority Classification

**AI Services MS2 | Supplementary Practice B | 60 Marks | Exactly 20 Evaluation Checks**

**Candidate notebook — complete the functions marked TODO.**

**Business task.** A customer support organization receives service tickets covering multiple products and communication channels. Each ticket contains numeric workload measurements, a creation timestamp, categorical metadata and, once reviewed, a priority level: HIGH, MEDIUM or LOW.

**Objective.** Build a multiclass classification notebook that normalizes priority labels, encodes three categories consistently, creates workload and time features, compares two classifiers using macro-F1 and weighted-F1, and returns readable priorities plus class probabilities for new tickets.

## 1. Imports, parameters and feature groups

In [1]:
import joblib
import numpy as np
import pandas as pd
from sklearn.base import BaseEstimator
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

TARGET = "priority"

CLASS_NAMES = ["HIGH", "LOW", "MEDIUM"]

RAW_NUMERIC = ["response_minutes", "touch_count", "backlog_count", "agent_count", "agent_experience_yrs"]

NUMERIC = RAW_NUMERIC + ["opened_hour", "opened_month", "backlog_per_agent"]

CATEGORICAL = ["channel", "product_area", "customer_tier"]

FEATURES = NUMERIC + CATEGORICAL

## Implementation contract — independently testable functions

**1. `load_data(filename: str) -> pd.DataFrame`**  
**Returns:** Return a cleaned DataFrame with ID, original fields, eight numeric features, normalized categories and optional priority.  
• Read CSV and preserve the first occurrence of each ticket_id. Parse opened_ts with mixed formats and remove invalid dates.  
• Coerce five raw numerics to numbers; replace both positive and negative infinity with NaN. Strip/lowercase the three categories, normalize x.0 codes with the supplied regex, and fill missing values with unknown.  
• Add opened_hour and opened_month from opened_ts. Compute backlog_per_agent using positive agent_count only; unsafe divisions become NaN.  
• If priority is present, normalize with astype("string").str.strip().str.upper().replace("", pd.NA).

In [ ]:
def load_data(filename: str) -> pd.DataFrame:
    """Normalize ticket records, calculate time/ratio features, and preserve optional target."""

    df = pd.read_csv(filename,dtype={'ticket_id':'string'})
    df = df.drop_duplicates(subset=['ticket_id'],keep='first')

    df['opened_ts'] = pd.to_datetime(df['opened_ts'],errors='coerce',dayfirst=True,format='mixed')
    df = df.dropna(subset=['opened_ts'])

    for col in RAW_NUMERIC:
        df[col] = pd.to_numeric(df[col],errors='coerce').replace([np.inf,-np.inf],np.nan)

    for col in CATEGORICAL:
        df[col] = (df[col].astype('string').str.strip().str.lower().str
        .replace(r"^([+-]?\d+)\.0+$", r"\1", regex=True)
        .replace("", pd.NA).fillna('unknown').astype(str))

    df['opened_hour'] = df['opened_ts'].dt.hour
    df['opened_month'] = df['opened_ts'].dt.month

    df['backlog_per_agent'] = (df['backlog_count']/df['agent_count'].where(df['agent_count']>0)).replace([np.inf,-np.inf],np.nan)

    if TARGET in df:
        df[TARGET] = df[TARGET].astype('string').str.strip().str.upper().replace("", pd.NA)

    return df.reset_index(drop=True)

**2. `build_pipeline(algorithm: BaseEstimator) -> Pipeline`**  
**Returns:** Return an unfitted pipeline containing ColumnTransformer and the supplied classifier.  
• Numeric branch: median imputation followed by StandardScaler for the eight numeric predictors.  
• Categorical branch: most-frequent imputation followed by OneHotEncoder(handle_unknown="ignore") for the three text categories.  
• Use separate branches and attach the estimator after ColumnTransformer.

In [ ]:
def build_pipeline(algorithm: BaseEstimator) -> Pipeline:
    """Return an unfitted mixed-type pipeline with numeric median and categorical mode imputation."""

    num = Pipeline([('imputer',SimpleImputer(strategy='median')),
                            ('scaler',StandardScaler())])

    cat = Pipeline(steps=[('imputer',SimpleImputer(strategy='most_frequent')),
                            ('encoding',OneHotEncoder(handle_unknown='ignore'))])

    preprocessor = ColumnTransformer([
        ('numeric',num,NUMERIC),
        ('categorical',cat,CATEGORICAL)
    ])

    return Pipeline([('preprocessor',preprocessor),('model',algorithm)])

**3. `prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, LabelEncoder]`**  
**Returns:** Return X_train, X_test, y_train, y_test, encoder (exactly five objects).  
• Use the common loader; filter both labeled files to HIGH, LOW and MEDIUM only. Fit a single LabelEncoder to the three permitted labels and reuse it to encode the held-back target.  
• Return eight numeric and three categorical predictor columns; do not include original timestamps, IDs or target.

In [ ]:
def prepare_datasets(train_file: str, test_file: str) -> tuple[pd.DataFrame, pd.DataFrame, pd.Series, pd.Series, LabelEncoder]:
    """Return X_train, X_test, integer y_train, integer y_test, fitted label encoder."""

    train_df , test_df = load_data(train_file) , load_data(test_file)
    train_df = train_df.loc[train_df[TARGET].isin(CLASS_NAMES)]
    test_df = test_df.loc[test_df[TARGET].isin(CLASS_NAMES)]

    encoder = LabelEncoder().fit(CLASS_NAMES)

    y_train = pd.Series(encoder.transform(train_df[TARGET]),index=train_df.index,name=TARGET)
    y_test = pd.Series(encoder.transform(test_df[TARGET]),index=test_df.index,name=TARGET)

    return train_df[FEATURES], test_df[FEATURES], y_train, y_test, encoder

**4. `train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]`**  
**Returns:** Return fitted logistic_regression and random_forest pipelines.  
• LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).  
• RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1).

In [ ]:
def train_models(X_train: pd.DataFrame, y_train: pd.Series) -> dict[str, Pipeline]:
    """Fit two independently preprocessed multiclass classifiers."""

    models = {
        'logistic_regression' : LogisticRegression(max_iter=1000,class_weight='balanced',random_state=42),
        'random_forrest' : RandomForestClassifier(n_estimators=80, max_depth=10, min_samples_leaf=2, class_weight="balanced", random_state=42, n_jobs=-1)
    }

    trained_models = {}

    for name,algo in models.items():
        pipeline = build_pipeline(algo)
        pipeline.fit(X_train,y_train)
        trained_models[name] = pipeline

    return trained_models

**5. `evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> tuple[pd.DataFrame, dict[str, np.ndarray]]`**  
**Returns:** Return (comparison, matrices).  
• Calculate accuracy, macro_f1 and weighted_f1 for each fitted model. Average macro-F1 equally across three classes; weighted-F1 uses test-set support. Apply zero_division=0.  
• Create a 3×3 confusion matrix for encoded class IDs 0,1,2, stored under each model name in matrices.  
• Return comparison columns model_name, accuracy, macro_f1, weighted_f1, ranked macro_f1 descending, weighted_f1 descending, model_name ascending. Do not fit or refit.

In [ ]:
def evaluate_models(trained_models: dict[str, Pipeline], X_test: pd.DataFrame, y_test: pd.Series) -> tuple[pd.DataFrame, dict[str, np.ndarray]]:
    """Compare accuracy, macro-F1, weighted-F1 and return confusion matrices for all models."""
    raise NotImplementedError('Complete evaluate_models')

**6. `select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]`**  
**Returns:** Return (best_name, best_model), retrieving existing top-ranked pipeline.  
• Use the first row of the sorted comparison and the matching object in trained_models.

In [ ]:
def select_best_model(comparison: pd.DataFrame, trained_models: dict[str, Pipeline]) -> tuple[str, Pipeline]:
    """Return first-ranked name and its existing fitted pipeline."""
    raise NotImplementedError('Complete select_best_model')

**7. `predict_priority(model: Pipeline, encoder: LabelEncoder, df: pd.DataFrame) -> pd.DataFrame`**  
**Returns:** Return ticket_id, predicted_priority, confidence, high_probability.  
• Predict encoded classes, use encoder.inverse_transform() to get readable priorities.  
• Set confidence to the largest class probability per row; set high_probability to the probability for the encoded HIGH class. Round both to 4 decimals. Keep IDs/order.

In [ ]:
def predict_priority(model: Pipeline, encoder: LabelEncoder, df: pd.DataFrame) -> pd.DataFrame:
    """Predict labels via inverse_transform, confidence and HIGH probability."""
    raise NotImplementedError('Complete predict_priority')

## Training and saved-model prediction

The training and prediction workflows are intentionally separate. Training fits and persists the chosen estimator with its preprocessing. Prediction must load the saved model and process new rows without calling `fit()`.

**`train() -> tuple[str, pd.DataFrame]`** — Return the winning model name and comparison report. Save a Joblib bundle containing both model and fitted LabelEncoder to best_model.pkl. Also create model_comparison.csv and the winning confusion_matrix.csv.

In [ ]:
def train() -> tuple[str, pd.DataFrame]:
    """Train and persist model+encoder, comparison and winning confusion matrix."""
    raise NotImplementedError('Complete train')

**`predict() -> pd.DataFrame`** — Return predictions for incoming tickets. Load saved model and encoder together; process ticket_predict.csv and write ticket_predictions.csv. Never call fit during prediction.

**Required output files:** `best_model.pkl`, `model_comparison.csv`, `confusion_matrix.csv`, `ticket_predictions.csv`

In [ ]:
def predict() -> pd.DataFrame:
    """Reload fitted model and label encoder, then write ticket_predictions.csv."""
    raise NotImplementedError('Complete predict')

## 4. Run the workflow

Execute the **training** call first, then the **prediction** call. You can also run these in separate notebook sessions after a model has been saved.

In [ ]:
# Uncomment to train after completing the functions:
# best_name, comparison = train()
# display(comparison)

In [ ]:
# Uncomment to predict after training:
# result = predict()
# display(result.head())